# Building LLM Applications with LangChain and Groq

## Objective
This notebook demonstrates the implementation of Large Language Model (LLM) applications using LangChain and Groq. It covers prompt engineering, chaining, tool usage, memory concepts, and vector databases.

## Technologies Used
- LangChain
- Groq API
- FAISS (Vector Store)
- HuggingFace Embeddings

In [2]:
!pip install -q langchain langchain-groq langchain-community faiss-cpu

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
google-adk 1.25.1 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 8.5.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [3]:
!pip install -q langchain-huggingface

## Importing Required Modules

In [4]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

## Loading API Key from Kaggle Secrets

In [5]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("GROQ_API_KEY")

os.environ["GROQ_API_KEY"] = api_key

print("API Loaded Successfully")

API Loaded Successfully


## Initializing Groq LLM Model

In [6]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.7
)

##  Basic LLM Invocation

In [7]:
response = llm.invoke("Explain Artificial Intelligence in simple terms")
print(response.content)

**What is Artificial Intelligence (AI)?**

Artificial Intelligence (AI) is a way to make machines think and learn like humans. It's a type of computer science that allows us to create programs that can:

1. **Learn**: Like a child, AI machines can learn from data, experiences, and feedback.
2. **Reason**: They can make decisions based on the information they've learned.
3. **Interact**: AI can communicate with humans and other machines in a more natural way.

**How does AI work?**

Imagine you're teaching a child to recognize pictures of cats and dogs. You show them many pictures and say "this is a cat" or "this is a dog." The child learns to recognize the differences between the two. AI machines work in a similar way:

1. **Data collection**: We gather large amounts of data, like pictures or text.
2. **Algorithms**: We use special computer programs (algorithms) to analyze the data and identify patterns.
3. **Training**: The AI machine learns from the data and patterns it's discovered.

##  Prompt Template for Structured Input

In [10]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "Explain {topic} in 3 simple points"
)

print(prompt.format(topic="Machine Learning"))

Explain Machine Learning in 3 simple points


##  Creating a Chain using LCEL (LangChain Expression Language)

In [11]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"topic": "Deep Learning"})
print(result)

Here are 3 simple points explaining Deep Learning:

1. **Inspired by the Brain**: Deep Learning is a subset of Machine Learning that mimics the structure and function of the human brain. It uses artificial neural networks (ANNs) with multiple layers to process and analyze complex data. These networks are made up of interconnected nodes (neurons) that learn and adapt through experience.

2. **Learning from Data**: In Deep Learning, computers are trained on large datasets to recognize patterns and relationships. The more data you feed the system, the more accurate it becomes at making predictions or decisions. This process is called supervised learning, where the computer is guided by labeled data to learn from.

3. **Automating Complex Tasks**: Deep Learning has made it possible to automate complex tasks that were previously difficult or impossible for computers to perform. Examples include:
- Image recognition (e.g., self-driving cars)
- Speech recognition (e.g., virtual assistants)
- 

##  Tool Implementation (Calculator)

In [12]:
def calculator(expr):
    try:
        return str(eval(expr))
    except:
        return "Error"

print("Calculator:", calculator("25 * 4"))

Calculator: 100


## Memory Concept (Basic Simulation)

In [13]:
memory = []

memory.append("My name is Sai")
memory.append("What is my name?")

print("Memory:", memory)

Memory: ['My name is Sai', 'What is my name?']


##  Vector Store using FAISS

In [14]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings()

texts = [
    "LangChain is powerful",
    "AI is transforming industries"
]

vectorstore = FAISS.from_texts(texts, embedding=embeddings)

results = vectorstore.similarity_search("What is AI?")
print(results)

/tmp/ipykernel_947/2946674614.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/tmp/ipykernel_947/2946674614.py:4: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Document(id='c34aca65-52c5-475d-a09c-413e750c690d', metadata={}, page_content='AI is transforming industries'), Document(id='4b6bcabe-46d6-4ed7-87e4-d5c854a9fe7a', metadata={}, page_content='LangChain is powerful')]


##  Final Integrated Pipeline

In [15]:
user_input = "Calculate 50 + 20"

# Tool execution
tool_result = calculator("50 + 20")

# LLM explanation
explanation = llm.invoke(f"Explain the result {tool_result}").content

print("User Input:", user_input)
print("Tool Result:", tool_result)
print("LLM Explanation:", explanation)

User Input: Calculate 50 + 20
Tool Result: 70
LLM Explanation: There are many possible results that can be associated with the number 70. Here are a few:

1. **Age**: 70 is a significant age milestone in many cultures, often associated with retirement, wisdom, and life experience.
2. **Percentage**: 70% is a common percentage used in various contexts, such as:
	* 70% of the human brain is made up of water.
	* 70% of the Earth's surface is covered in water.
	* 70% of the world's population uses the internet.
3. **Sports**: In baseball, a 70-win season is often considered a successful one for a team.
4. **Numerology**: In numerology, 70 is often associated with spiritual growth, introspection, and self-awareness.
5. **Mathematics**: There are 70 prime numbers less than 200.
6. **Music**: There are many songs and albums with the title "70" or referencing the number 70.
7. **Year**: 1970 was a significant year in history, with major events such as the Apollo 13 mission and the Kent State s

## Simulated Agent Workflow (Decision + Tool + LLM)

In [16]:
# Simulated Agent Workflow (decision + tool + reasoning)

user_query = "What is 15 * 3?"

# Step 1: Decide if tool needed
if "*" in user_query or "+" in user_query:
    tool_result = calculator("15 * 3")
    final_response = llm.invoke(
        f"The result is {tool_result}. Explain this simply."
    ).content
else:
    final_response = llm.invoke(user_query).content

print("Agent Decision Output:", final_response)

Agent Decision Output: There's not enough information to provide a complete answer. Can you give more context or details about what you're trying to explain or solve?
